[Reference](https://pub.towardsai.net/langgraph-101-what-every-ai-engineer-should-know-ea73ba38f0a5)

In [1]:
pip install langgraph langchain-openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 14.6 MB/s eta 0:00:00


In [2]:
# state
from typing import TypedDict

class GraphState(TypedDict):
    question: str
    answer: str

In [4]:
# nodes
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")


def generate(state: dict) -> dict:
    question = state["question"]
    response = llm.invoke(f"Answer the following: {question}")
    return {"answer": response.content}

In [6]:
# app
from langgraph.graph import StateGraph

builder = StateGraph(GraphState)
builder.add_node("generate", generate)
builder.set_entry_point("generate")


graph = builder.compile()

output = graph.invoke({"question": "What is LangGraph?"})
print(output)

In [7]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is sunny ☀️"

In [8]:
tools = [get_weather]

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4")

def decide_topic(state: dict) -> str:
    question = state["question"].lower()
    return "weather" if "weather" in question else "general"

def use_weather_tool(state: dict) -> dict:
    return {"answer": get_weather.invoke("Paris")}

def generate(state: dict) -> dict:
    question = state["question"]
    response = llm.invoke(f"Answer: {question}")
    return {"answer": response.content}

In [10]:
from langgraph.graph import StateGraph, END

builder = StateGraph(dict)

builder.add_node("decide", lambda x: x)  # Acts as passthrough
builder.add_node("weather", use_weather_tool)
builder.add_node("generate", generate)

builder.set_entry_point("decide")

builder.add_conditional_edges(
    "decide",
    decide_topic,
    {
        "weather": "weather",
        "general": "generate"
    }
)

builder.add_edge("weather", END)
builder.add_edge("generate", END)

graph = builder.compile()

In [11]:
print(graph.invoke({"question": "What's the weather in Paris?"})["answer"])
print(graph.invoke({"question": "Explain LangGraph in simple words."})["answer"])